# 11. Jupyter on CIRRUS

← [Specialized workloads](10-workloads.ipynb) · next: [Functions](12-functions.ipynb)

<https://jupyter.k8s.ucar.edu/> — UCAR network or VPN, UCAR CIT credentials.
**Anyone with a CIT account already has access.** No ticket, no namespace, no
`kubectl`.

This is the lowest-effort thing on the whole platform and the one most worth
knowing about, because it turns "I need a machine with a GPU and xarray" into a
browser tab.

### Choosing a server

The Server Options page after login offers:

* **NSF NCAR CPU notebooks**, in three resource tiers.
* **NSF NCAR GPU notebooks** — PyTorch and TensorFlow images, with kernels
  `cirrus-pytorch-base` and `cirrus-tensorflow-base` already populated. GPUs are
  shared using NVIDIA **time slicing**, so several people run on one physical GPU
  concurrently; it is fair-shared rather than exclusive, which is the right
  trade-off for interactive work and the wrong one for a benchmark.
* **Custom environment** — a community scientific image, your own image, or a
  repository built on demand.

The base environments are `cirrus-base` (Python, assembled from user input and
maintained by the CCPP team) and `r-4.4` (mirroring the package set on the HPC
JupyterHub).

### Your own environment, three ways

**A conda environment that persists.** `/home/jovyan` is a persistent volume, so
an environment created under `/home/jovyan/my-conda-envs` survives between
sessions and appears in the kernel menu — `nb_conda_kernels` does that
automatically:

```bash
conda env create -f environment.yml
conda activate my-env
```


**Binder, from a repository.** *Choose Your Environment → Build your own image*,
give it a repository URL and optionally a branch, tag or commit. If the repo has
an `environment.yml` or `requirements.txt`, Binder builds a container from it and
launches JupyterLab inside it, with your JupyterHub session's resources and home
directory. For sharing with people who do not have a UCAR account, see
[public Binder](#sharing-publicly-binder) below.

**Your own image**, if you have already built one for Harbor
([page 2](02-containers.ipynb)) — the Custom Environment option takes an image
reference.

### Scaling out with Dask

One notebook is one pod, so `LocalCluster` is limited by the resources you asked
for at launch. Start there anyway:

```python
from dask.distributed import Client, LocalCluster

cluster = LocalCluster("cluster-name", n_workers=4)
client = Client(cluster)
client
```


When that is not enough, **`dask-gateway`** provisions workers as their own pods
with their own resources:

```python
from dask.distributed import Client
from dask_gateway import GatewayCluster

cluster = GatewayCluster(
    "http://traefik-dask-gateway/services/dask-gateway/",
    public_address="https://jupyter.k8s.ucar.edu/services/dask-gateway/",
    auth="jupyterhub",
)
cluster.adapt(minimum=2, maximum=20)      # scale with the work
client = Client(cluster)
client
```


Two things to hold on to:

* **`dask-jobqueue` does not work here.** It provisions through PBS or Slurm, and
  there is no batch scheduler on CIRRUS. `PBSCluster` belongs on the HPC
  JupyterHub; `dask-gateway` is its counterpart on Kubernetes. Mixing them up is
  the most common Dask question on this platform.
* **`cluster.close()` when you are done.** Idle clusters are reaped
  automatically, but workers you have forgotten are quota nobody else can use.

The dashboard URL printed by the client works with the Dask JupyterLab extension
in the sidebar — paste it in and you can tile the task stream and worker panels
beside the notebook, which is genuinely the fastest way to find out why a
computation is slow.

Docs: <https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/cirrus/guides/06-jupyter-on-cirrus/jupyterhub/>

### Dask for an application, not a notebook

Everything above lives and dies with your notebook session. A dashboard that
computes against Dask needs workers that are always there. Two ways to get them:

* **A dedicated cluster in your own chart.** A scheduler and workers as ordinary
  Deployments next to your web application, which is what
  [`dask-helm`](https://github.com/NCAR/cirrus-examples/tree/main/helm/dask-helm)
  in `cirrus-examples` sets up. It is the same pattern as pages 3–5, with three
  Deployments instead of one.
* **The shared Dask Gateway**, from an application rather than a notebook. It
  authenticates through JupyterHub and needs an API key. That key is a
  credential, so it reaches your application the
  [page 6](06-secrets.ipynb) way: stored in OpenBao, delivered by an
  ExternalSecret.

---

## Sharing publicly: Binder

JupyterHub needs a UCAR account. For a tutorial, a paper's reproducible figure,
or a "launch" button in a Jupyter Book, the public **BinderHub** at
<https://binder.k8s.ucar.edu/> needs nothing at all: give it a repository, and
it builds the environment from the files already there (`environment.yml`,
`requirements.txt`, a `Dockerfile`) and opens JupyterLab in the browser.

Know its limits before you hand out a link. Sessions get 1 CPU and 2 GB of
memory, nothing is saved when they end, and anyone can use it — so only point it
at repositories and data you are happy to make public. For anything heavier, use
JupyterHub.

Docs: <https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/cirrus/guides/15-binder/binder/>

---

## Check yourself

1. You want a GPU and xarray for an afternoon's exploration. What is the
   fastest route, and what do you have to request?
2. Why does `dask-jobqueue` not work on CIRRUS, and what replaces it?
3. Your dashboard needs Dask workers that are always running. Which two options
   do you have, and which one needs a secret?
4. You want workshop attendees without UCAR accounts to run your notebooks. What
   do you send them, and what should you check about the repository first?

---

← [10. Specialized workloads](10-workloads.ipynb) · next: [12. Functions](12-functions.ipynb)